# Build Termux Libraries for CodeMaster AI Studio

This notebook builds the Termux AAR libraries needed for the embedded terminal.
Run all cells, then download the AAR files and place them in `app/libs/`

In [ ]:
!apt-get update -q && apt-get install -q -y openjdk-17-jdk git wget unzip
!java -version

In [ ]:
# Clone termux-app with sparse checkout for library modules only
!git clone --depth 1 --filter=blob:none --sparse https://github.com/termux/termux-app.git termux-app
%cd termux-app
!git sparse-checkout set terminal-emulator terminal-view termux-shared gradle

In [ ]:
# Setup Android SDK
!mkdir -p /root/android-sdk/cmdline-tools
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
!unzip -q commandlinetools-linux-11076708_latest.zip -d /root/android-sdk/cmdline-tools
!mv /root/android-sdk/cmdline-tools/cmdline-tools /root/android-sdk/cmdline-tools/latest

import os
os.environ['ANDROID_HOME'] = '/root/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/root/android-sdk'

!yes | /root/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
!mkdir -p /root/android-sdk/build-tools /root/android-sdk/platforms
!/root/android-sdk/cmdline-tools/latest/bin/sdkmanager 'build-tools;34.0.0' 'platforms;android-34' 'platform-tools'

In [ ]:
# Create local.properties
with open('local.properties', 'w') as f:
    f.write('sdk.dir=/root/android-sdk\n')

# Build the library modules
!chmod +x gradlew
!./gradlew :terminal-emulator:assembleRelease :terminal-view:assembleRelease :termux-shared:assembleRelease --no-daemon

In [ ]:
# List and create download links for AAR files
import os
from google.colab import files

aar_files = [
    'terminal-emulator/build/outputs/aar/terminal-emulator-release.aar',
    'terminal-view/build/outputs/aar/terminal-view-release.aar',
    'termux-shared/build/outputs/aar/termux-shared-release.aar'
]

for aar in aar_files:
    if os.path.exists(aar):
        print(f'✓ Found: {aar}')
        files.download(aar)
    else:
        print(f'✗ Missing: {aar}')

## Next Steps:
1. After downloading the AAR files, place them in `codemaster-ai-studio/app/libs/`
2. Uncomment the Termux dependencies in `app/build.gradle.kts`
3. Rebuild your app